# CAGE 4: CPU Rollout + CUDA PPO
Select an NVIDIA GPU runtime. Run the smoke test before the queue. This notebook keeps the existing learning protocol; it does not resume old CPU runs.


In [ ]:
!nvidia-smi


## Upload source
Upload `dist/acd_colab_source.tar.gz` generated from the current repository. The archive excludes runs and virtual environments.


In [ ]:
from google.colab import files
from pathlib import Path
import tarfile
import os

assert not Path("/content/acd_benchmark").exists(), "Source folder already exists; inspect before replacing."
uploaded = files.upload()
assert len(uploaded) == 1, "Upload exactly one source archive."
archive_path = next(iter(uploaded))
with tarfile.open(archive_path, "r:gz") as archive:
    archive.extractall("/content", filter="data")
os.chdir("/content/acd_benchmark")
print(Path.cwd())


## Locked environment
Use Python 3.10 and the project's locked dependencies. No change to Colab's global Torch is required.


In [ ]:
%pip install -q uv
import subprocess
subprocess.run(["uv", "sync", "--locked", "--python", "3.10", "--group", "dev"], check=True)


## CUDA qualification
This fails if no CUDA GPU is available. All three encoders must pass CPU-worker/CUDA-learner and end-to-end tests.


In [ ]:
subprocess.run(["uv", "run", "--no-sync", "python", "-m", "scripts.experiments.smoke_cuda"], check=True)


## Persistent output
Drive keeps completed checkpoints and logs after a runtime disconnect. Exact training resume is not yet supported. Keep the same storage policy for all timed runs.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## One training seed, six sequential cells
Set SEED to 42, 43 or 44. Do not compare hardware costs across different GPU/CPU allocations. A complete seed can still outlast a Colab session; inspect throughput before committing to the full queue.


In [ ]:
SEED = 42
RUN_ROOT = f"/content/drive/MyDrive/acd_runs/encoder_retention_500k_cuda_seed{SEED}"
command = [
    "uv", "run", "--no-sync", "python", "-u", "-m",
    "scripts.experiments.run_encoder_matrix",
    "--config", "configs/experiments/encoder_retention_500k_cuda.json",
    "--run-root", RUN_ROOT, "--seeds", str(SEED), "--foreground",
]
subprocess.run(command, check=True)


## Summary
Use stochastic evaluation per q as the primary result, greedy as secondary. Missing training seeds must remain explicit.


In [ ]:
subprocess.run([
    "uv", "run", "--no-sync", "python", "-m", "scripts.analysis.matrix_status",
    "--run-root", RUN_ROOT,
], check=True)
